In [ ]:
!pip -q install -U datasets transformers accelerate scikit-learn iterative-stratification psutil

print("Installation complete.")

from collections import Counter

from google.colab import drive
drive.mount('/content/drive')

# Load the dataset WITHOUT downloading it again
from datasets import load_from_disk

dataset = load_from_disk(
    "/content/drive/MyDrive/multieurlex_all_languages"
)

# Get the label feature from the training split
label_feature = dataset["train"].features["labels"].feature

# Count every label occurrence across all splits
for split in dataset:
    label_counts = Counter()

    for labels in dataset[split]["labels"]:
        label_counts.update(labels)

    print(f"\n{split.upper()}")
    print("Documents:", dataset[split].num_rows)
    print("Unique labels used:", len(label_counts))
    print("Total label assignments:", sum(label_counts.values()))

# Number of possible EUROVOC label classes
print("\nTotal possible label classes:", len(label_feature.names))


In [ ]:
# @title
import os
import time
import random
import numpy as np
import pandas as pd
import torch
import json

from accelerate import Accelerator # Import Accelerator

# 'FP32' accelerator
accelerator = Accelerator(mixed_precision='no')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ---- Main experiment controls ----
MAX_LENGTH = 256
BATCH_SIZE = 16
NUM_EPOCHS = 10
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
THRESHOLD = 0.50
N_TRAIN_PARTS = 10 # Number of chunks to divide the training data into, to get more stable training

# Choose one of the three transformer models tested
#MODEL_NAME= "bert-based-uncased"
#MODEL_NAME = "bert-base-multilingual-cased"   # mBERT
MODEL_NAME = "xlm-roberta-base"              # XLM-R

print("Model:", MODEL_NAME)

# Determine total training samples
total_training_samples = len(dataset["train"])

# Calculate total training steps and warmup steps
total_steps = (total_training_samples // BATCH_SIZE) * NUM_EPOCHS
warmup_steps = int(0.1 * total_steps)

print(f"Total training samples: {total_training_samples}")
print(f"Total training steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")


In [ ]:
#Evaluation Metrics
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

def multilabel_metrics(y_true, y_pred):
    return {
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "micro_precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "exact_match": accuracy_score(y_true, y_pred)
    }

In [ ]:
TEXT_COLUMN = "text"
LABEL_COLUMN = "labels"
# Build a consistent label vocabulary directly from the dataset's label feature.
# This assumes `label_feature` is already defined (e.g., from cell `l578f8uhvqEu`).

# Get the list of all possible label names (EUROVOC IDs as strings)
all_label_names_from_dataset = label_feature.names
NUM_LABELS = len(all_label_names_from_dataset)

# Create a simple 0-based index mapping for the multi-hot vector
# where the key is the label's integer ID from the dataset, and value is its index in our vector
label_to_index = {i: i for i in range(NUM_LABELS)}

print("Number of labels:", NUM_LABELS)
print("First mappings (dataset label ID -> internal vector index):", list(label_to_index.items())[:10])

def multi_hot(label_ids, mapping):
    vector = np.zeros(len(mapping), dtype=np.float32)
    for label_id in label_ids:
        vector[mapping[int(label_id)]] = 1.0
    return vector


In [ ]:
from functools import partial
from transformers import AutoTokenizer 
from datasets import concatenate_datasets

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Function to apply tokenization and multi-hot encoding
def preprocess_function(examples, tokenizer, label_to_index, max_length):
    # Extract English text from the dictionary for tokenization, for training and validation purposes
    texts_to_tokenize = [example.get('en', '') for example in examples[TEXT_COLUMN]]


    tokenized_inputs = tokenizer(
        texts_to_tokenize,
        truncation=True,
        max_length=max_length,
        padding=False 
    )

    # Convert labels to multi-hot vectors
    multi_hot_labels = [
        multi_hot(
            [lbl for lbl in row if int(lbl) in label_to_index], 
            label_to_index
        )
        for row in examples[LABEL_COLUMN]
    ]
    tokenized_inputs["labels"] = multi_hot_labels
    return tokenized_inputs

# Prepare the partial function for mapping
preprocess_map_fn = partial(preprocess_function, tokenizer=tokenizer, label_to_index=label_to_index, max_length=MAX_LENGTH)

# In this approach, to get more stable RAM usage, 
# We split the training (55k) dataset into 10 chunks of 5k documents.
# Therefore, Over the 10 Epochs, there are total 100 chunks of 5k documents. 
train_split_size = len(dataset["train"])
part_size = train_split_size // N_TRAIN_PARTS

# Process validation data
tokenized_val = dataset["validation"].map(
    preprocess_map_fn,
    batched=True,
    remove_columns=[TEXT_COLUMN, LABEL_COLUMN],
    load_from_cache_file=False
)
print(f"Tokenized validation data size: {len(tokenized_val)}")

# Update tokenized_dataset to directly use raw 'train' and hold pre-tokenized val/test
tokenized_dataset = {
    "train": dataset["train"], # Store the raw training dataset
    "validation": tokenized_val,
}

print("\nTokenized dataset features (from validation split, as train is raw):")
print(tokenized_dataset["validation"].column_names)

In [ ]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

# Data collator will dynamically pad the batches
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True)

# Remove 'celex_id' column as it's not needed for model input and causes tensor conversion errors
val_dataset_for_loader = tokenized_dataset["validation"].remove_columns(["celex_id"])

# Create DataLoaders for validation set
val_loader = DataLoader(
    val_dataset_for_loader,
    batch_size=BATCH_SIZE,
    shuffle=False, # No need to shuffle validation data
    collate_fn=data_collator
)

print("Validation Test DataLoader created.")

Validation and Test DataLoaders created.
labels torch.Size([16, 567]) torch.int64
input_ids torch.Size([16, 256]) torch.int64
attention_mask torch.Size([16, 256]) torch.int64


In [ ]:
from transformers import AutoConfig, AutoModelForSequenceClassification

config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification"
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    config=config
)

# model.to(device) # Accelerator handles device placement


In [ ]:
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW

# Initialize optimizer
optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

# Prepare model, optimizer, val_loader, test_loader, scheduler, and data_collator once with Accelerator
# Training loaders will be prepared dynamically in the loop
model, optimizer, val_loader, scheduler, data_collator = accelerator.prepare(
    model, optimizer, val_loader, scheduler, data_collator
)


print("Training steps:", total_steps)
print("Warmup steps:", warmup_steps)

In [ ]:
import numpy as np
from tqdm.auto import tqdm
from scipy.special import expit 

def evaluate_model(model, loader, threshold=0.5):
    # Use the unwrapped model for evaluation if it's been wrapped by Accelerator
    unwrapped_model = accelerator.unwrap_model(model)
    unwrapped_model.eval()

    all_logits = []
    all_labels = []
    total_loss = 0.0
    n_batches = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating", leave=False):
            # Data is already on the correct device after accelerator.prepare
            # batch = {k: v.to(device) for k, v in batch.items()}

            # Convert labels to float type before passing to model
            batch["labels"] = batch["labels"].to(torch.float32)

            outputs = unwrapped_model(**batch) # Use unwrapped_model for evaluation

            total_loss += outputs.loss.item()
            n_batches += 1

            all_logits.append(outputs.logits.detach().cpu())
            all_labels.append(batch["labels"].detach().cpu())

    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()

    probabilities = expit(logits)
    predictions = (probabilities >= threshold).astype(int)

    metrics = multilabel_metrics(labels, predictions)
    metrics["loss"] = total_loss / max(n_batches, 1)

    return metrics, probabilities, predictions, labels

### Removed: Empty Cell

This cell was previously empty and has been removed to streamline the notebook for mBERT training.

In [ ]:
import numpy as np
import time
from tqdm.auto import tqdm
import gc # For garbage collection
import torch.cuda # Import for clearing cuda cache

# History and best_val_f1 are initialized in VQsJZGmTjRTv and loaded from checkpoint if resuming

training_start = time.perf_counter()

print(f"Total training samples: {total_training_samples}")
print(f"Total steps for scheduler: {total_steps}")
print(f"Warmup steps for scheduler: {warmup_steps}")
print(f"Starting training from epoch {start_epoch + 1} for {NUM_EPOCHS - start_epoch} epochs.")

# Calculate chunking parameters here, ensure they are accessible
# N_TRAIN_PARTS is already defined in VQsJZGmTjRTv and available
train_split_size = len(tokenized_dataset["train"]) # Now points to original raw dataset["train"]
part_size = train_split_size // N_TRAIN_PARTS

# Loop through epochs, starting from start_epoch if resuming
for epoch in tqdm(range(start_epoch, NUM_EPOCHS), desc="Epochs", initial=start_epoch, total=NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    current_step_in_epoch = 0 # To track total steps across parts for averaging loss

    epoch_start = time.perf_counter()

    # Iterate through training parts dynamically
    for part_idx in range(N_TRAIN_PARTS):
        # Select the raw chunk from the original dataset
        start_idx = part_idx * part_size
        end_idx = (part_idx + 1) * part_size if part_idx < N_TRAIN_PARTS - 1 else train_split_size

        print(f"  Epoch {epoch+1}/{NUM_EPOCHS}: Tokenizing and training on part {part_idx+1}/{N_TRAIN_PARTS} (indices {start_idx} to {end_idx-1})...")

        raw_train_part = tokenized_dataset["train"].select(range(start_idx, end_idx))

        # Tokenize the current part on-the-fly
        tokenized_part = raw_train_part.map(
            preprocess_map_fn, # preprocess_map_fn is from Y37rJtc3i3fh
            batched=True,
            remove_columns=[TEXT_COLUMN, LABEL_COLUMN, "celex_id"],
            load_from_cache_file=False
        )

        # Create DataLoader for the current tokenized training part
        train_loader_part = DataLoader(
            tokenized_part,
            batch_size=BATCH_SIZE,
            shuffle=True, # Shuffle each part
            collate_fn=data_collator # Use the data collator
        )

        # Prepare this specific part's loader with accelerator
        prepared_train_loader_part = accelerator.prepare(train_loader_part)

        for batch_idx, batch in enumerate(tqdm(prepared_train_loader_part, desc=f"Training part {part_idx+1}/{N_TRAIN_PARTS}", leave=False)):
            optimizer.zero_grad()

            # Convert labels to float type before passing to model
            batch["labels"] = batch["labels"].to(torch.float32)

            # --- Diagnostic: Print labels dtype ---
            #print(f"    Labels dtype before model call: {batch['labels'].dtype}")

            outputs = model(**batch)
            loss = outputs.loss

            accelerator.backward(loss)
            optimizer.step()
            scheduler.step() # Scheduler step is per batch

            running_loss += loss.item()
            current_step_in_epoch += 1

        # Explicitly delete objects from current part to free RAM
        del raw_train_part
        del tokenized_part
        del train_loader_part
        del prepared_train_loader_part
        torch.cuda.empty_cache() # Clear GPU cache if applicable
        gc.collect() # Force garbage collection
        print(f"  Memory freed after part {part_idx+1}.")


    train_loss = running_loss / max(current_step_in_epoch, 1) # Average loss over all batches from all parts

    # Evaluate using the evaluate_model function, which handles unwrapping
    # Note: val_loader and test_loader are prepared once.
    val_metrics, _, _, _ = evaluate_model(
        model,
        val_loader,
        threshold=THRESHOLD
    )

    epoch_time = time.perf_counter() - epoch_start
    current_lr = scheduler.get_last_lr()[0]
    print(f"[Diagnostic] LR at end of epoch {epoch+1}: {current_lr:.3e}")
    record = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_metrics["loss"],
        "val_micro_f1": val_metrics["micro_f1"],
        "val_macro_f1": val_metrics["macro_f1"],
        "epoch_time_sec": epoch_time
    }
    history.append(record)

    _diag_model = accelerator.unwrap_model(model)
    _diag_model.eval()
    _fixed_batch = next(iter(val_loader))
    with torch.no_grad():
        _fixed_logits = _diag_model(
            input_ids=_fixed_batch["input_ids"],
            attention_mask=_fixed_batch["attention_mask"]
        ).logits.detach().cpu()
    if 'prev_epoch_logits' in dir():
        _identical = torch.equal(_fixed_logits, prev_epoch_logits)
        _max_diff = (_fixed_logits - prev_epoch_logits).abs().max().item()
        print(f"[Diagnostic] Logits identical to previous epoch: {_identical}")
        print(f"[Diagnostic] Max abs logit diff vs previous epoch: {_max_diff:.2e}")
    prev_epoch_logits = _fixed_logits
    _diag_model.train()


    current_val_macro_f1 = val_metrics["macro_f1"]
    if current_val_macro_f1 > best_val_f1:
        best_val_f1 = current_val_macro_f1
        # The model is saved at the end of every epoch, so we track best_val_f1
        # in the checkpoint but don't need a separate best_state for the model weights.

    print(
        f"Epoch {epoch+1}/{NUM_EPOCHS} | "
        f"train loss={train_loss:.10f} | "
        f"val loss={val_metrics['loss']:.10f} | "
        f"val micro-F1={val_metrics['micro_f1']:.10f} | "
        f"val macro-F1={val_metrics['macro_f1']:.10f} | "
        f"time={epoch_time:.1f}s"
    )

    # --- Checkpoint Saving Logic (Atomic Save) ---
    
    # Save to a temporary file first using accelerator.save
    accelerator.save(checkpoint_to_save, TEMP_CHECKPOINT_PATH)
    # Then atomically replace the main checkpoint file
    os.replace(TEMP_CHECKPOINT_PATH, CHECKPOINT_PATH)
    print(f"Checkpoint saved to {CHECKPOINT_PATH} after epoch {epoch+1}.")

total_training_time = time.perf_counter() - training_start # Total training time across all resumed sessions

print(f"Total Transformer training time: {total_training_time/60:.2f} minutes")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd # Import pandas

# Convert history list of dicts to a DataFrame for easier plotting
history_df = pd.DataFrame(history)

plt.figure(figsize=(8, 4))
plt.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Train loss")
plt.plot(history_df["epoch"], history_df["val_loss"], marker="o", label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history_df["epoch"], history_df["val_micro_f1"], marker="o", label="Validation micro-F1")
plt.plot(history_df["epoch"], history_df["val_macro_f1"], marker="o", label="Validation macro-F1")
plt.xlabel("Epoch")
plt.ylabel("F1")
plt.title("Validation F1 by Epoch")
plt.legend()
plt.show()

In [ ]:
from tqdm.auto import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import numpy as np
import torch
import pandas as pd
import seaborn as sns


LANGUAGES = ['bg', 'cs', 'da', 'de', 'el', 'en', 'es', 'et', 'fi', 'fr', 'hr',
             'hu', 'it', 'lt', 'lv', 'mt', 'nl', 'pl', 'pt', 'ro', 'sk', 'sl', 'sv']

# ============================================================
# STEP 1 — Run inference ONCE across all 23 languages
# Produces raw probabilities + true labels per language.
# Everything else below is derived from this without re-running the model.
# ============================================================

def evaluate_multilingual_full(model, tokenizer, raw_dataset, label_to_index,
                                 num_labels, languages, batch_size=16, max_length=256):
    model.eval()
    probs_by_lang, true_by_lang = {}, {}

    for lang in tqdm(languages, desc="Languages"):
        texts, label_rows = [], []
        for row in raw_dataset:
            text = row["text"].get(lang)
            if text:
                texts.append(text)
                mh = np.zeros(num_labels, dtype=np.float32)
                for l in row["labels"]:
                    if l in label_to_index:
                        mh[label_to_index[l]] = 1.0
                label_rows.append(mh)
        if not texts:
            tqdm.write(f"  Skipping {lang}: no documents found.")
            continue

        logits_list = []
        n_batches = (len(texts) + batch_size - 1) // batch_size
        for i in tqdm(range(0, len(texts), batch_size), total=n_batches, desc=f"  {lang}", leave=False):
            enc = tokenizer(texts[i:i+batch_size], truncation=True, padding=True,
                             max_length=max_length, return_tensors="pt").to(accelerator.device)
            with torch.no_grad():
                out = model(**enc)
            logits_list.append(out.logits.cpu())

        probs_by_lang[lang] = torch.sigmoid(torch.cat(logits_list)).numpy()
        true_by_lang[lang] = np.stack(label_rows)
        tqdm.write(f"  {lang}: n={len(texts)} done.")

    return probs_by_lang, true_by_lang


# Run it (replace `model`, `tokenizer`, `dataset["test"]`, `label_to_index`, `NUM_LABELS`
# with your actual variables)
probs_by_lang, true_by_lang = evaluate_multilingual_full(
    accelerator.unwrap_model(model), tokenizer, dataset["test"],
    label_to_index, NUM_LABELS, LANGUAGES
)

# Save immediately so you never lose this even if the model/session is gone later
np.savez(
    "/content/drive/MyDrive/eval_probs_by_lang.npz",
    **{f"{l}_probs": probs_by_lang[l] for l in probs_by_lang},
    **{f"{l}_true": true_by_lang[l] for l in true_by_lang}
)
print("Saved raw probabilities + labels to Drive.\n")


# ============================================================
# STEP 2 — Per-language metrics (fixed threshold = 0.5)
# ============================================================

def build_per_lang_df(probs_by_lang, true_by_lang, threshold=0.5):
    results = []
    for lang, probs in probs_by_lang.items():
        preds = (probs >= threshold).astype(int)
        y_true = true_by_lang[lang]
        results.append({
            "language": lang,
            "n_examples": len(probs),
            "micro_precision": precision_score(y_true, preds, average="micro", zero_division=0),
            "micro_recall": recall_score(y_true, preds, average="micro", zero_division=0),
            "micro_f1": f1_score(y_true, preds, average="micro", zero_division=0),
            "macro_precision": precision_score(y_true, preds, average="macro", zero_division=0),
            "macro_recall": recall_score(y_true, preds, average="macro", zero_division=0),
            "macro_f1": f1_score(y_true, preds, average="macro", zero_division=0),
        })
    return pd.DataFrame(results)

per_lang_df = build_per_lang_df(probs_by_lang, true_by_lang, threshold=0.5)
print("=== Per-language metrics (threshold = 0.5) ===")
display(per_lang_df.round(4))


# ============================================================
# STEP 3 — Overall metrics: POOLED and AVERAGED
# ============================================================

pooled_probs = np.concatenate(list(probs_by_lang.values()))
pooled_true = np.concatenate(list(true_by_lang.values()))
pooled_preds = (pooled_probs >= 0.5).astype(int)

pooled_overall = {
    "micro_precision": precision_score(pooled_true, pooled_preds, average="micro", zero_division=0),
    "micro_recall": recall_score(pooled_true, pooled_preds, average="micro", zero_division=0),
    "micro_f1": f1_score(pooled_true, pooled_preds, average="micro", zero_division=0),
    "macro_precision": precision_score(pooled_true, pooled_preds, average="macro", zero_division=0),
    "macro_recall": recall_score(pooled_true, pooled_preds, average="macro", zero_division=0),
    "macro_f1": f1_score(pooled_true, pooled_preds, average="macro", zero_division=0),
    "exact_match": accuracy_score(pooled_true, pooled_preds),
}

print("\n=== Overall metrics — POOLED across all 23 languages (threshold = 0.5) ===")
display(pd.Series(pooled_overall).round(4))

overall_avg = per_lang_df[[
    "micro_precision", "micro_recall", "micro_f1",
    "macro_precision", "macro_recall", "macro_f1"
]].mean()

print("\n=== Overall metrics — AVERAGED across the 23 per-language scores (threshold = 0.5) ===")
display(overall_avg.round(4))


# ============================================================
# STEP 4 — Threshold sweep (pooled across all 23 languages)
# ============================================================

thresholds = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
threshold_results = []

for t in thresholds:
    preds = (pooled_probs >= t).astype(int)
    threshold_results.append({
        "threshold": t,
        "micro_precision": precision_score(pooled_true, preds, average="micro", zero_division=0),
        "micro_recall": recall_score(pooled_true, preds, average="micro", zero_division=0),
        "micro_f1": f1_score(pooled_true, preds, average="micro", zero_division=0),
        "macro_f1": f1_score(pooled_true, preds, average="macro", zero_division=0),
    })

threshold_df = pd.DataFrame(threshold_results)
print("\n=== Threshold sweep (pooled across all 23 languages) ===")
display(threshold_df.round(4))

# --- Plots ---
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
plt.plot(threshold_df["threshold"], threshold_df["micro_precision"], marker="o", label="Precision")
plt.plot(threshold_df["threshold"], threshold_df["micro_recall"], marker="o", label="Recall")
plt.plot(threshold_df["threshold"], threshold_df["micro_f1"], marker="o", label="F1")
plt.xlabel("Sigmoid threshold")
plt.ylabel("Score")
plt.title("Threshold vs Precision / Recall / F1 (23 languages pooled)")
plt.legend()
plt.show()

heatmap_data = threshold_df.set_index("threshold")[["micro_precision", "micro_recall", "micro_f1"]].T
plt.figure(figsize=(10, 3))
sns.heatmap(heatmap_data, annot=True, fmt=".3f", cmap="viridis")
plt.xlabel("Threshold")
plt.ylabel("Metric")
plt.title("Threshold vs Precision / Recall / F1 (Heatmap, pooled)")
plt.show()

In [ ]:
# Parameter count
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

if torch.cuda.is_available():
    print(f"Peak GPU memory: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")


In [ ]:
# Measure inference time on the test loader
model.eval()

if torch.cuda.is_available():
    torch.cuda.synchronize()

start = time.perf_counter()

# Use the prepared test_loader for inference
for batch in test_loader:
    # accelerator handles placing batch on device
    with torch.no_grad():
        _ = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )

if torch.cuda.is_available():
    torch.cuda.synchronize()

inference_time = time.perf_counter() - start

print(f"Test inference time: {inference_time:.2f} sec")
print(f"Documents/sec: {len(tokenized_dataset['test'])/inference_time:.2f}")

Test inference time: 18.70 sec
Documents/sec: 267.37


In [ ]:
### Label-wise F1 Analysis (Transformer Only)

transformer_label_f1 = f1_score(
    test_labels, transformer_pred, average=None, zero_division=0
)

# Create a DataFrame for label-wise transformer F1 scores
label_analysis_df = pd.DataFrame({
    "label_index": range(NUM_LABELS),
    "transformer_f1": transformer_label_f1,
    "support": test_labels.sum(axis=0).astype(int) # Using test_labels for support
})

print("Top 15 Transformer F1 Scores by Label:")
display(
    label_analysis_df.sort_values("transformer_f1", ascending=False).head(15)
)

print("Bottom 15 Transformer F1 Scores by Label:")
display(
    label_analysis_df.sort_values("transformer_f1").head(15)
)

# The previous cell previously compared label-wise F1 scores between the baseline and transformer models.
# Since the baseline model is no longer being trained and its predictions are unavailable,
# the comparison aspect has been removed, and this cell now focuses on the transformer's performance.

Top 15 Transformer F1 Scores by Label:


,label_index,transformer_f1,support
78,78,0.990576,481
197,197,0.949206,315
423,423,0.943215,543
108,108,0.934524,178
518,518,0.923077,6
419,419,0.920726,543
79,79,0.916667,23
505,505,0.915152,1016
528,528,0.912471,664
455,455,0.911504,239


Bottom 15 Transformer F1 Scores by Label:


,label_index,transformer_f1,support
307,307,0.0,3
325,325,0.0,10
326,326,0.0,9
327,327,0.0,0
328,328,0.0,4
329,329,0.0,0
330,330,0.0,3
331,331,0.0,0
332,332,0.0,55
333,333,0.0,75
